# Homework 2

## Question 1

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu126
CUDA runtime: 12.6
GPU available: False


In [28]:
import copy
import librosa 
import numpy as np
from IPython.display import Audio, display

import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [3]:
clean, sr = librosa.load("data/train_clean_male.wav", sr=None)
noisy, sr_noisy = librosa.load("data/train_dirty_male.wav", sr=None)

print("Clean shape:", clean.shape)
print("Noisy shape:", noisy.shape)
print("Sample rate:", sr)
print("Duration:", len(clean) / sr, "seconds")

Clean shape: (1258899,)
Noisy shape: (1258899,)
Sample rate: 16000
Duration: 78.6811875 seconds


In [20]:
val_noisy, sr_val = librosa.load("data/test_x_01.wav", sr=None)
val_clean, sr_clean = librosa.load("data/test_s_01.wav", sr=None)

print(sr_val == sr_clean == sr)

X_val = librosa.stft(val_noisy, n_fft=1024, hop_length=512)
print("Validation STFT shape:", X_val.shape)

True
Validation STFT shape: (513, 142)


In [4]:
print("Clean speech:")
display(Audio(clean, rate=sr))

print("Noisy speech:")
display(Audio(noisy, rate=sr_noisy))

Clean speech:


Noisy speech:


In [5]:
S = librosa.stft(clean, n_fft=1024, hop_length=512)
X = librosa.stft(noisy, n_fft=1024, hop_length=512)

print("Clean spectrogram shape:", S.shape)
print("Noisy spectrogram shape:", X.shape)

Clean spectrogram shape: (513, 2459)
Noisy spectrogram shape: (513, 2459)


In [ ]:
S_mag = np.abs(S)
X_mag = np.abs(X)

print(S_mag.shape)
print(X_mag.shape)

X_train = X_mag.T # transposing to get training examples by features (frequency magnitudes) 
Y_train = S_mag.T

print(X_train.shape)
print(Y_train.shape)

(513, 2459)
(513, 2459)
(2459, 513)
(2459, 513)


In [ ]:
# TEST CELL - inspecting shapes to determine how to normalize 

print("Noisy magnitudes:")
print("Min:", X_train.min())
print("Max:", X_train.max())
print("Mean:", X_train.mean())
print("Std:", X_train.std())

print("\nClean magnitudes:")
print("Min:", Y_train.min())
print("Max:", Y_train.max())
print("Mean:", Y_train.mean())
print("Std:", Y_train.std())

Noisy magnitudes:
Min: 4.824803e-06
Max: 31.603909
Mean: 0.1137142
Std: 0.3290241

Clean magnitudes:
Min: 4.001008e-07
Max: 31.5746
Mean: 0.067518465
Std: 0.30736867


In [ ]:
scale = X_train.std()

X_train_scaled = X_train / scale # not subracting by mean because we need to preserve non negatives
Y_train_scaled = Y_train / scale

print("Scale factor:", scale)
print("Scaled input mean:", X_train_scaled.mean())
print("Scaled input std:", X_train_scaled.std())
print("Scaled target mean:", Y_train_scaled.mean())

Scale factor: 0.3290241
Scaled input mean: 0.34561062
Scaled input std: 1.0
Scaled target mean: 0.20520827


In [15]:
class SpeechDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(513, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 513),
            nn.Softplus() # soft plus to make sure the network does not predict negatives 
        )

    def forward(self, x):
        return self.network(x)

In [21]:
def reconstruct_audio(predicted_mag, noisy_stft, length):
    phase = noisy_stft / (np.abs(noisy_stft) + 1e-10)
    reconstructed_stft = predicted_mag * phase
    reconstructed_audio = librosa.istft(reconstructed_stft, hop_length=512, length=length)
    return reconstructed_audio 

In [22]:
# TEST CELL

reconstructed_noisy = reconstruct_audio(
    predicted_mag=np.abs(X_val),
    noisy_stft=X_val,
    length=len(val_noisy)
)

print("Original shape:", val_noisy.shape)
print("Reconstructed shape:", reconstructed_noisy.shape)

print(
    "Reconstruction MSE:",
    np.mean((val_noisy - reconstructed_noisy) ** 2)
)

Original shape: (72619,)
Reconstructed shape: (72619,)
Reconstruction MSE: 7.335114e-18


In [23]:
def calculate_snr(clean, reconstructed):
    signal_energy = np.sum(clean ** 2)
    error_energy = np.sum((clean - reconstructed) ** 2)

    snr = 10 * np.log10(
        (signal_energy + 1e-20) / (error_energy + 1e-20)
    )

    return snr

In [24]:
# TEST CELL
print("Clean samples:", len(val_clean))
print("Noisy samples:", len(val_noisy))

baseline_snr = calculate_snr(val_clean, val_noisy)

print(f"Baseline noisy SNR: {baseline_snr:.2f} dB")


Clean samples: 72619
Noisy samples: 72619
Baseline noisy SNR: 13.23 dB


In [ ]:
# TEST CELL - checking to make sur shapes match

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SpeechDenoiser().to(device)

sample_batch = torch.tensor(
    X_train_scaled[:32],
    dtype=torch.float32,
    device=device
)

predictions = model(sample_batch)

print("device:", device)
print("input shape:", sample_batch.shape)
print("output shape:", predictions.shape)
print("Minimum predicted magnitude:", predictions.min().item())


device: cpu
input shape: torch.Size([32, 513])
output shape: torch.Size([32, 513])
Minimum predicted magnitude: 0.42928096652030945


In [26]:
def validate_model(model, X_val, val_clean, val_noisy, scale, device):
    model.eval()

    X_val_mag = np.abs(X_val)
    X_val_scaled = X_val_mag.T / scale

    X_val_tensor = torch.tensor(
        X_val_scaled,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():
        predicted_scaled = model(X_val_tensor)

    predicted_mag = predicted_scaled.cpu().numpy().T * scale
    reconstructed = reconstruct_audio(predicted_mag, X_val, length=len(val_noisy))
    snr = calculate_snr(val_clean, reconstructed)

    return snr

In [27]:
# TEST CELL

initial_snr = validate_model(
    model,
    X_val,
    val_clean,
    val_noisy,
    scale,
    device
)

print(f"Untrained model SNR: {initial_snr:.2f} dB")
print(f"Baseline SNR: 13.23 dB")

Untrained model SNR: 0.05 dB
Baseline SNR: 13.23 dB


In [29]:
X_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
Y_tensor = torch.tensor(Y_train_scaled, dtype=torch.float32)

train_dataset = TensorDataset(X_tensor, Y_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [30]:
# num_epochs = 100
# patience = 15

# best_snr = float("-inf")
# best_weights = None
# best_epoch = 0
# epochs_without_impovement = 0

# train_losses = []
# val_snrs = []

# for epoch in range(num_epochs):
#     model.train()
#     total_loss = 0.0

#     for noisy_batch, clean_batch in train_loader:
#         noisy_batch = noisy_batch.to(device)
#         clean_batch = clean_batch.to(device)

#         optimizer.zero_grad()

#         predictions = model(noisy_batch)
#         loss = criterion(predictions, clean_batch)

#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item() * noisy_batch.size(0)

#     avg_loss = total_loss / len(train_dataset)
#     train_losses.append(avg_loss)

#     val_snr = validate_model(model, X_val, val_clean, val_noisy, scale, device)
#     val_snrs.append(val_snr)

#     if val_snr > best_snr:
#         best_snr = val_snr
#         best_epoch = epoch + 1
#         best_weights = copy.deepcopy(model.state_dict())
#         epochs_without_impovement = 0
#     else:
#         epochs_without_impovement += 1

#     if (epoch + 1) % 5 == 0 or epoch == 0:
#         print(
#             f"Epoch {epoch+1:3d} | "
#             f"Train MSE: {avg_loss:.5f} | "
#             f"Validation SNR: {val_snr:.2f} dB | "
#             f"Best: {best_snr:.2f} dB"
#         )

#     if epochs_without_impovement >= patience:
#         print(f"\nEarly stopping at epoch {epoch + 1}")
#         break

# model.load_state_dict(best_weights)

# print(f"\nBest epoch: {best_epoch}")
# print(f"Best validation SNR: {best_snr:.2f} dB")
# print(f"Imporovement over baseline: {best_snr - baseline_snr:.2f} dB")